# Pocket-Agent Training Notebook

Fine-tune **Qwen3-0.6B-Instruct** with QLoRA for structured tool calling.

**Runtime:** GPU → T4 (Runtime > Change runtime type > T4 GPU)

**Total time:** ~75-90 minutes end-to-end
- Setup: 5 min
- Data gen: 15-25 min (with API) or 2 min (rule-based only)
- Training: 35-45 min (500 steps)
- Quantize + eval: 10 min

## 0. Setup

In [ ]:
# Clone repo
!git clone https://github.com/YOUR_USERNAME/pocket-agent.git
%cd pocket-agent

In [ ]:
# Install training dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl datasets transformers accelerate peft bitsandbytes openai

In [1]:
!pip install -q trl datasets transformers accelerate peft bitsandbytes openai

In [ ]:
# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

## 1. Generate Training Data

Set your OpenAI API key for GPT-4o-mini teacher, or use `--no-api` for free rule-based generation.

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # Replace with your key, or leave empty for --no-api

In [ ]:
# With API (recommended — better refusals and adversarial examples)
if os.environ.get('OPENAI_API_KEY', '').startswith('sk-'):
    !python data/generate_data.py --count 1500 --out data/training_data.jsonl
else:
    print("No API key set — using rule-based generation (free, slightly less diverse)")
    !python data/generate_data.py --no-api --count 1100 --out data/training_data.jsonl

In [ ]:
# Quick sanity check
!wc -l data/training_data.jsonl
!python -c "
import json
lines = open('data/training_data.jsonl').readlines()
ex = json.loads(lines[0])
print(f'Example 0: {len(ex[\"messages\"])} messages')
for m in ex['messages']:
    print(f'  [{m[\"role\"]}]: {m[\"content\"][:80]}')
"

## 2. Fine-tune with Unsloth QLoRA

In [ ]:
!python train/finetune.py \
    --data data/training_data.jsonl \
    --output artifacts/adapter \
    --model unsloth/Qwen3-0.6B \
    --max-steps 500 \
    --lr 2e-4 \
    --batch-size 4 \
    --grad-accum 4 \
    --lora-r 16 \
    --lora-alpha 16

## 3. Export to GGUF Q4_K_M

In [ ]:
!python quantize.py --adapter artifacts/adapter --out artifacts --quant q4_k_m

In [ ]:
# Check model size and gate compliance
!ls -lh artifacts/*.gguf
!python -c "
import json
from pathlib import Path
gguf = list(Path('artifacts').glob('*.gguf'))
if gguf:
    mb = gguf[0].stat().st_size / 1024**2
    print(f'Size: {mb:.1f} MB')
    print('✓ Passes ≤500 MB gate' if mb <= 500 else '✗ EXCEEDS 500 MB gate')
    print('✓ ≤250 MB bonus!' if mb <= 250 else f'Not ≤250 MB ({mb:.1f} MB)')
"

## 4. Evaluate on Public Test Set

First, install CPU inference deps.

In [ ]:
!pip install -q llama-cpp-python gradio

In [ ]:
# Run evaluation (requires data/public_test.jsonl from starter pack)
!python eval/evaluate.py --test-file data/public_test.jsonl --verbose

## 5. Quick Inference Test

In [ ]:
from inference import run
import json

test_cases = [
    ("What's the weather in Paris in Celsius?", []),
    ("Convert 100 USD to EUR", []),
    ("Schedule 'Team meeting' for 2025-06-15", []),
    ("Tell me a joke", []),  # should refuse
    ("What's 100 USD in GBP?", []),
    ("Now convert that to JPY", [{"role": "user", "content": "What's 100 USD in GBP?"}, {"role": "assistant", "content": '<tool_call>\n{"tool": "currency", "args": {"amount": 100, "from": "USD", "to": "GBP"}}\n</tool_call>'}]),
]

for prompt, history in test_cases:
    print(f'User: {prompt}')
    resp = run(prompt, history)
    print(f'Model: {resp}')
    print()

## 6. Launch Gradio Demo

In [ ]:
# Launch with public sharing link (required for Colab)
!python demo.py --share

## 7. Upload Artifacts to Hugging Face Hub (Optional)

In [ ]:
# Optional: push adapter to HF Hub for grader access
# !pip install -q huggingface_hub
# from huggingface_hub import HfApi
# api = HfApi(token='hf_...')
# api.upload_folder(folder_path='artifacts/', repo_id='YOUR_USERNAME/pocket-agent', repo_type='model')

## Gate Check Summary

In [ ]:
!make check-gates
!make check-inference-imports